In [ ]:
import os
import cv2
import json
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import timm
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    balanced_accuracy_score, confusion_matrix, 
    classification_report, roc_auc_score, roc_curve, auc,
    precision_recall_curve, f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {gpu_name} ({gpu_mem:.1f} GB)")

In [ ]:
@dataclass
class Config:
    # ==================== PATHS ====================
    CSV_PATH: str = "/kaggle/input/isic-2019-task-1/ISIC_2019_5folds_metadata.csv"
    IMG_ROOT: str = "/kaggle/input/isic-2019-task-1/cropped_lesions/cropped_lesions"
    OUTPUT_DIR: str = "/kaggle/working"
    TEST_CSV_PATH: str = "/kaggle/input/isic-2019-task-1/ISIC_2019_test_metadata.csv"
    TEST_IMG_ROOT: str = "/kaggle/input/isic-2019-task-1/cropped_lesions_testset/cropped_lesions_testset"
    
    # ==================== MODEL ====================
    # DenseNet121-CBAM: ~8.5M params (DenseNet121 + CBAM modules)
    MODEL_NAME: str = "densenet121"
    N_CLASSES: int = 8
    PRETRAINED: bool = True
    DROP_RATE: float = 0.3
    
    # CBAM specific
    CBAM_REDUCTION: int = 16  # Channel reduction ratio
    CBAM_KERNEL_SIZE: int = 7  # Spatial attention kernel size
    
    # ==================== TRAINING ====================
    IMG_SIZE: int = 256
    N_FOLDS: int = 5
    BATCH_SIZE: int = 64
    EPOCHS: int = 100
    MIN_EPOCHS: int = 15
    PATIENCE: int = 10
    
    # ==================== OPTIMIZER ====================
    BASE_LR: float = 3e-4
    MIN_LR: float = 1e-6
    WEIGHT_DECAY: float = 5e-4
    WARMUP_EPOCHS: int = 5
    
    # ==================== REGULARIZATION ====================
    LABEL_SMOOTHING: float = 0.1
    
    # Mixup/CutMix
    USE_MIXUP: bool = True
    MIXUP_ALPHA: float = 0.4
    CUTMIX_ALPHA: float = 0.4
    MIXUP_PROB: float = 0.3
    MIXUP_SWITCH_PROB: float = 0.5
    
    # ==================== LOSS ====================
    USE_FOCAL_LOSS: bool = False
    
    # ==================== CLASS BALANCING ====================
    USE_CLASS_BALANCED_SAMPLING: bool = True
    MALIGNANT_BOOST: float = 1.5
    SCC_EXTRA_BOOST: float = 2.0
    
    # ==================== HARDWARE ====================
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int = 4
    PIN_MEMORY: bool = True
    
    # ==================== CLASS INFO ====================
    CLASS_NAMES: tuple = ('MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC')
    MALIGNANT_INDICES: tuple = (0, 2, 7)

cfg = Config()

# Print config
print("="*60)
print("CONFIGURATION: DenseNet121-CBAM")
print("="*60)
print(f"Model: {cfg.MODEL_NAME} + CBAM")
print(f"Parameters: ~8.5M (DenseNet121 + CBAM)")
print(f"CBAM Reduction: {cfg.CBAM_REDUCTION}")
print(f"CBAM Kernel Size: {cfg.CBAM_KERNEL_SIZE}")
print(f"Image Size: {cfg.IMG_SIZE}")
print(f"Batch Size: {cfg.BATCH_SIZE}")
print(f"Learning Rate: {cfg.BASE_LR}")
print(f"Weight Decay: {cfg.WEIGHT_DECAY}")
print(f"Drop Rate: {cfg.DROP_RATE}")
print(f"Label Smoothing: {cfg.LABEL_SMOOTHING}")
print(f"Use Mixup/CutMix: {cfg.USE_MIXUP} (α={cfg.MIXUP_ALPHA}, prob={cfg.MIXUP_PROB})")
print(f"Use Class-balanced Sampling: {cfg.USE_CLASS_BALANCED_SAMPLING}")
print(f"Malignant Boost: {cfg.MALIGNANT_BOOST}x, SCC Extra: {cfg.SCC_EXTRA_BOOST}x")
print(f"Epochs: {cfg.EPOCHS}")
print(f"Device: {cfg.DEVICE}")
print("="*60)

In [ ]:
# ==================== CBAM: Convolutional Block Attention Module ====================
# Paper: https://arxiv.org/abs/1807.06521

class ChannelAttention(nn.Module):
    """
    Channel Attention Module.
    Applies attention across channels using both max and avg pooling.
    """
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        # Shared MLP
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        attention = self.sigmoid(avg_out + max_out)
        return x * attention


class SpatialAttention(nn.Module):
    """
    Spatial Attention Module.
    Applies attention across spatial dimensions using channel-wise pooling.
    """
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv(concat))
        return x * attention


class CBAM(nn.Module):
    """
    CBAM: Convolutional Block Attention Module.
    Combines Channel Attention and Spatial Attention.
    """
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


print("✓ CBAM modules defined")
print("  - ChannelAttention: Applies attention across channels")
print("  - SpatialAttention: Applies attention across spatial dimensions")
print("  - CBAM: Combines both for comprehensive attention")

In [ ]:
class DenseNet121CBAM(nn.Module):
    """
    DenseNet121 with CBAM attention modules.
    CBAM is added after each Dense Block for enhanced feature learning.
    
    Architecture:
    - DenseNet121 backbone (pretrained)
    - CBAM after each transition layer
    - Custom classifier head with dropout
    """
    def __init__(self, num_classes=8, pretrained=True, drop_rate=0.3, 
                 reduction=16, kernel_size=7):
        super(DenseNet121CBAM, self).__init__()
        
        # Load pretrained DenseNet121
        self.backbone = timm.create_model(
            'densenet121', 
            pretrained=pretrained, 
            num_classes=0,  # Remove classifier
            global_pool=''  # Remove global pooling
        )
        
        # DenseNet121 feature channels after each dense block:
        # DenseBlock1: 256 -> Transition1: 128
        # DenseBlock2: 512 -> Transition2: 256
        # DenseBlock3: 1024 -> Transition3: 512
        # DenseBlock4: 1024 (final features)
        
        # CBAM modules for each stage
        # After transition layers (128, 256, 512) and final (1024)
        self.cbam1 = CBAM(128, reduction=reduction, kernel_size=kernel_size)
        self.cbam2 = CBAM(256, reduction=reduction, kernel_size=kernel_size)
        self.cbam3 = CBAM(512, reduction=reduction, kernel_size=kernel_size)
        self.cbam4 = CBAM(1024, reduction=reduction, kernel_size=kernel_size)
        
        # Get the number of features from backbone
        self.num_features = 1024
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(p=drop_rate),
            nn.Linear(self.num_features, num_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize CBAM and classifier weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if m.weight.requires_grad:
                    nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward_features(self, x):
        """Forward through backbone with CBAM attention"""
        # Initial convolution
        x = self.backbone.features.conv0(x)
        x = self.backbone.features.norm0(x)
        x = self.backbone.features.pool0(x)
        
        # DenseBlock1 + Transition1
        x = self.backbone.features.denseblock1(x)
        x = self.backbone.features.transition1(x)
        x = self.cbam1(x)  # CBAM after transition1 (128 channels)
        
        # DenseBlock2 + Transition2
        x = self.backbone.features.denseblock2(x)
        x = self.backbone.features.transition2(x)
        x = self.cbam2(x)  # CBAM after transition2 (256 channels)
        
        # DenseBlock3 + Transition3
        x = self.backbone.features.denseblock3(x)
        x = self.backbone.features.transition3(x)
        x = self.cbam3(x)  # CBAM after transition3 (512 channels)
        
        # DenseBlock4 (final)
        x = self.backbone.features.denseblock4(x)
        x = self.backbone.features.norm5(x)
        x = self.cbam4(x)  # CBAM after denseblock4 (1024 channels)
        
        return x
    
    def forward(self, x):
        x = self.forward_features(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        x = self.classifier(x)
        return x


def create_model(cfg, use_dp=True):
    """Create DenseNet121-CBAM model"""
    model = DenseNet121CBAM(
        num_classes=cfg.N_CLASSES,
        pretrained=cfg.PRETRAINED,
        drop_rate=cfg.DROP_RATE,
        reduction=cfg.CBAM_REDUCTION,
        kernel_size=cfg.CBAM_KERNEL_SIZE
    )
    model = model.to(cfg.DEVICE)
    
    if use_dp and torch.cuda.device_count() > 1:
        print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    return model


def get_scheduler(optimizer, cfg, num_train_steps):
    """Warmup linear rồi CosineAnnealing"""
    warmup_steps = cfg.WARMUP_EPOCHS * (num_train_steps // cfg.EPOCHS)
    
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, num_train_steps - warmup_steps))
        return max(cfg.MIN_LR / cfg.BASE_LR, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Test model
print("\nTesting model creation...")
test_model = create_model(cfg, use_dp=False)
n_params = sum(p.numel() for p in test_model.parameters())
n_trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Model: DenseNet121-CBAM")
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

with torch.no_grad():
    dummy_input = torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).to(cfg.DEVICE)
    output = test_model(dummy_input)
    print(f"Output shape: {output.shape}")

del test_model
torch.cuda.empty_cache()
print("✓ DenseNet121-CBAM model factory working correctly!")

In [ ]:
# Đọc CSV
df = pd.read_csv(cfg.CSV_PATH)
print(f"Dataset size: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['target'].value_counts().sort_index())

# Tính class weight (inverse frequency) với MALIGNANT BOOST
counts = df["target"].value_counts().sort_index().values
N = len(df)
weights = N / counts
weights = weights / weights.mean()

# Áp dụng Malignant Boost
print(f"\n{'='*50}")
print("APPLYING MALIGNANT CLASS BOOST")
print(f"{'='*50}")

original_weights = weights.copy()
for idx in cfg.MALIGNANT_INDICES:
    weights[idx] *= cfg.MALIGNANT_BOOST
    print(f"  {cfg.CLASS_NAMES[idx]}: {original_weights[idx]:.4f} -> {weights[idx]:.4f} (x{cfg.MALIGNANT_BOOST})")

# SCC Extra Boost
scc_idx = 7
weights[scc_idx] *= cfg.SCC_EXTRA_BOOST
print(f"  SCC extra boost: {weights[scc_idx]/cfg.SCC_EXTRA_BOOST:.4f} -> {weights[scc_idx]:.4f} (x{cfg.SCC_EXTRA_BOOST})")

class_weight = torch.tensor(weights, dtype=torch.float).to(cfg.DEVICE)

print(f"\nFinal Class weights:")
for i, (name, w) in enumerate(zip(cfg.CLASS_NAMES, weights)):
    malignant_marker = "⚠️ MALIGNANT" if i in cfg.MALIGNANT_INDICES else ""
    print(f"  {i}: {name} = {w:.4f} {malignant_marker}")

# Fold distribution
print(f"\nFold distribution:")
print(df['fold'].value_counts().sort_index())

In [ ]:
def get_train_transforms(img_size):
    """Augmentation cho DenseNet121-CBAM"""
    return A.Compose([
        A.RandomResizedCrop(
            size=(img_size, img_size), 
            scale=(0.7, 1.0),
            ratio=(0.9, 1.1),
            p=1.0
        ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Transpose(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1, 
            scale_limit=0.15, 
            rotate_limit=90,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5
        ),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
        ], p=0.3),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.OneOf([
            A.OpticalDistortion(distort_limit=0.05, p=1.0),
            A.GridDistortion(distort_limit=0.1, p=1.0),
        ], p=0.2),
        A.CoarseDropout(
            max_holes=8, max_height=img_size//8, max_width=img_size//8,
            min_holes=4, min_height=img_size//16, min_width=img_size//16,
            fill_value=0, p=0.5
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


print("✓ Augmentation functions defined")

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, df, transforms=None, img_root=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.img_root = img_root
        self.is_test = is_test
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, os.path.basename(row["path"]))
        image = cv2.imread(img_path)
        
        if image is None:
            raise ValueError(f"Cannot read image: {img_path}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented["image"]
        
        if self.is_test:
            return image, row['image']
        
        target = torch.tensor(row["target"], dtype=torch.long)
        return image, target


print("✓ ISICDataset class defined")

In [ ]:
class WeightedSoftTargetCrossEntropy(nn.Module):
    """SoftTargetCrossEntropy với class weights để giữ Malignant Boost khi dùng Mixup."""
    def __init__(self, class_weights=None):
        super().__init__()
        self.class_weights = class_weights
    
    def forward(self, x, target):
        log_probs = F.log_softmax(x, dim=-1)
        
        if self.class_weights is not None:
            weights = self.class_weights.to(x.device)
            weighted_target = target * weights.unsqueeze(0)
            weighted_target = weighted_target / (weighted_target.sum(dim=-1, keepdim=True) + 1e-8) * target.sum(dim=-1, keepdim=True)
            loss = -torch.sum(weighted_target * log_probs, dim=-1)
        else:
            loss = -torch.sum(target * log_probs, dim=-1)
        
        return loss.mean()


def setup_mixup(cfg, class_weight_tensor):
    """Setup Mixup/CutMix"""
    if not cfg.USE_MIXUP:
        return None, None
    
    mixup_fn = Mixup(
        mixup_alpha=cfg.MIXUP_ALPHA,
        cutmix_alpha=cfg.CUTMIX_ALPHA,
        cutmix_minmax=None,
        prob=cfg.MIXUP_PROB,
        switch_prob=cfg.MIXUP_SWITCH_PROB,
        mode='batch',
        label_smoothing=cfg.LABEL_SMOOTHING,
        num_classes=cfg.N_CLASSES
    )
    
    mixup_criterion = WeightedSoftTargetCrossEntropy(class_weights=class_weight_tensor)
    
    print("✓ Mixup/CutMix enabled")
    print(f"  Mixup α: {cfg.MIXUP_ALPHA}")
    print(f"  CutMix α: {cfg.CUTMIX_ALPHA}")
    print(f"  Apply Prob: {cfg.MIXUP_PROB}")
    
    return mixup_fn, mixup_criterion


# Criterion setup
criterion = nn.CrossEntropyLoss(weight=class_weight, label_smoothing=cfg.LABEL_SMOOTHING)
print(f"✓ Using CrossEntropyLoss with class_weight and label_smoothing={cfg.LABEL_SMOOTHING}")

# Setup Mixup
mixup_fn, mixup_criterion = setup_mixup(cfg, class_weight)

In [ ]:
def train_one_fold(fold, df, cfg, criterion, mixup_fn, mixup_criterion):
    print(f"\n{'='*60}")
    print(f"FOLD {fold} - DenseNet121-CBAM")
    print(f"{'='*60}")
    
    # Data preparation
    train_df = df[df.fold != fold].reset_index(drop=True)
    val_df = df[df.fold == fold].reset_index(drop=True)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples: {len(val_df)}")
    
    train_dataset = ISICDataset(train_df, get_train_transforms(cfg.IMG_SIZE), cfg.IMG_ROOT)
    val_dataset = ISICDataset(val_df, get_val_transforms(cfg.IMG_SIZE), cfg.IMG_ROOT)
    
    # Class-balanced sampling
    if cfg.USE_CLASS_BALANCED_SAMPLING:
        train_counts = train_df['target'].value_counts().sort_index().values
        train_sample_weights = 1.0 / train_counts[train_df['target'].values]
        sampler = WeightedRandomSampler(
            weights=train_sample_weights,
            num_samples=len(train_sample_weights),
            replacement=True
        )
        train_loader = DataLoader(
            train_dataset, batch_size=cfg.BATCH_SIZE,
            sampler=sampler,
            num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY, drop_last=True
        )
        print(f"✓ Using Class-balanced Sampling")
    else:
        train_loader = DataLoader(
            train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
            num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY, drop_last=True
        )
    
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.BATCH_SIZE * 2, shuffle=False,
        num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY
    )
    
    # Model & Optimizer
    model = create_model(cfg, use_dp=True)
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=cfg.BASE_LR, 
        weight_decay=cfg.WEIGHT_DECAY
    )
    
    num_train_steps = len(train_loader) * cfg.EPOCHS
    scheduler = get_scheduler(optimizer, cfg, num_train_steps)
    scaler = GradScaler()
    
    # Training state
    best_bal_acc = 0.0
    best_val_loss = float('inf')
    epochs_no_improve = 0
    checkpoint_path = os.path.join(cfg.OUTPUT_DIR, f"densenet121_cbam_fold{fold}_best.pth")
    
    oof_preds = []
    oof_targets = []
    oof_images = []
    
    history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'bal_acc': [], 'learning_rate': []}
    
    # Training loop
    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS} [Train]")
        for images, targets in pbar:
            images = images.to(cfg.DEVICE)
            targets = targets.to(cfg.DEVICE)
            
            # Apply Mixup/CutMix
            if mixup_fn is not None:
                images, targets_mixed = mixup_fn(images, targets)
                current_criterion = mixup_criterion
            else:
                targets_mixed = targets
                current_criterion = criterion
            
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(images)
                loss = current_criterion(outputs, targets_mixed)
            
            scaler.scale(loss).backward()
            
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.2e}"})
        
        avg_train_loss = train_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Validation
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []
        all_probs = []
        
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS} [Val]"):
                images = images.to(cfg.DEVICE)
                targets = targets.to(cfg.DEVICE)
                
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, targets)
                
                val_loss += loss.item()
                probs = torch.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)
                
                all_probs.append(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        all_probs = np.vstack(all_probs)
        bal_acc = balanced_accuracy_score(all_targets, all_preds)
        
        history['epoch'].append(epoch)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['bal_acc'].append(bal_acc)
        history['learning_rate'].append(current_lr)
        
        gap = avg_train_loss - avg_val_loss
        overfit_warning = "⚠️ Overfitting!" if gap < -0.3 else ""
        
        print(f"\nEpoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}, bal_acc={bal_acc:.4f} {overfit_warning}")
        
        # Early stopping
        if bal_acc > best_bal_acc + 1e-3:
            best_bal_acc = bal_acc
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            
            model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'best_bal_acc': best_bal_acc,
                'best_val_loss': best_val_loss,
            }, checkpoint_path)
            print(f"✓ Saved best model: bal_acc={best_bal_acc:.4f}, val_loss={best_val_loss:.4f}")
            
            oof_preds = all_probs
            oof_targets = all_targets
            oof_images = val_df['image'].tolist()
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s)")
        
        if epoch >= cfg.MIN_EPOCHS and epochs_no_improve >= cfg.PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
    
    del model
    torch.cuda.empty_cache()
    
    return {
        'fold': fold,
        'best_bal_acc': best_bal_acc,
        'best_val_loss': best_val_loss,
        'checkpoint_path': checkpoint_path,
        'oof_preds': oof_preds,
        'oof_targets': oof_targets,
        'oof_images': oof_images,
        'history': history,
    }

In [ ]:
# Chạy training
results = []
all_oof = []
all_histories = []

for fold in range(cfg.N_FOLDS):
    fold_result = train_one_fold(fold, df, cfg, criterion, mixup_fn, mixup_criterion)
    results.append(fold_result)
    all_histories.append(fold_result['history'])
    
    fold_oof = pd.DataFrame({
        'image': fold_result['oof_images'],
        'fold': fold,
        'target': fold_result['oof_targets'],
        **{f'prob_{i}': fold_result['oof_preds'][:, i] for i in range(cfg.N_CLASSES)}
    })
    all_oof.append(fold_oof)

# Summary
bal_accs = [r['best_bal_acc'] for r in results]
print(f"\n{'='*60}")
print(f"RESULTS SUMMARY - DenseNet121-CBAM")
print(f"{'='*60}")
for i, acc in enumerate(bal_accs):
    print(f"Fold {i}: {acc:.4f}")
print(f"Mean: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
print(f"{'='*60}")

In [ ]:
# Concat OOF predictions
oof_df = pd.concat(all_oof, ignore_index=True)
oof_df.to_csv(os.path.join(cfg.OUTPUT_DIR, 'oof_densenet121_cbam.csv'), index=False)

# Calculate metrics
oof_preds_class = oof_df[[f'prob_{i}' for i in range(cfg.N_CLASSES)]].values.argmax(axis=1)
oof_bal_acc = balanced_accuracy_score(oof_df['target'], oof_preds_class)

print(f"\nOOF Balanced Accuracy: {oof_bal_acc:.4f}")
print("\nClassification Report:")
print(classification_report(oof_df['target'], oof_preds_class, target_names=cfg.CLASS_NAMES))

# Per-class ROC-AUC
print("\nPer-class ROC-AUC:")
for i, class_name in enumerate(cfg.CLASS_NAMES):
    y_true_binary = (oof_df['target'] == i).astype(int)
    y_prob = oof_df[f'prob_{i}'].values
    try:
        auc_score = roc_auc_score(y_true_binary, y_prob)
        malignant = "⚠️ MALIGNANT" if class_name in ['MEL', 'BCC', 'SCC'] else ""
        print(f"  {class_name}: {auc_score:.4f} {malignant}")
    except:
        print(f"  {class_name}: N/A")

cm = confusion_matrix(oof_df['target'], oof_preds_class)
print(f"\n✓ OOF evaluation complete")

In [ ]:
def find_optimal_thresholds(oof_df, cfg, strategy='f1'):
    print(f"\n{'='*60}")
    print(f"THRESHOLD OPTIMIZATION (Strategy: {strategy})")
    print(f"{'='*60}")
    
    optimal_thresholds = {}
    threshold_metrics = {}
    
    prob_cols = [f'prob_{i}' for i in range(cfg.N_CLASSES)]
    probs = oof_df[prob_cols].values
    targets = oof_df['target'].values
    
    for i, class_name in enumerate(cfg.CLASS_NAMES):
        y_true_binary = (targets == i).astype(int)
        y_prob = probs[:, i]
        
        precision, recall, thresholds = precision_recall_curve(y_true_binary, y_prob)
        precision = precision[:-1]
        recall = recall[:-1]
        
        if len(thresholds) == 0:
            best_threshold, best_f1, best_precision, best_recall = 0.5, 0.0, 0.0, 0.0
        elif strategy == 'f1':
            f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
            best_idx = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx]
            best_f1 = f1_scores[best_idx]
            best_precision = precision[best_idx]
            best_recall = recall[best_idx]
        elif strategy == 'recall':
            target_recall = 0.85 if i in cfg.MALIGNANT_INDICES else 0.70
            valid_idx = np.where(recall >= target_recall)[0]
            if len(valid_idx) > 0:
                best_idx = valid_idx[np.argmax(precision[valid_idx])]
            else:
                best_idx = np.argmax(recall)
            best_threshold = thresholds[best_idx]
            best_f1 = 2 * precision[best_idx] * recall[best_idx] / (precision[best_idx] + recall[best_idx] + 1e-8)
            best_precision = precision[best_idx]
            best_recall = recall[best_idx]
        else:
            diff = np.abs(precision - recall)
            best_idx = np.argmin(diff)
            best_threshold = thresholds[best_idx]
            best_f1 = 2 * precision[best_idx] * recall[best_idx] / (precision[best_idx] + recall[best_idx] + 1e-8)
            best_precision = precision[best_idx]
            best_recall = recall[best_idx]
        
        optimal_thresholds[class_name] = float(best_threshold)
        threshold_metrics[class_name] = {
            'threshold': float(best_threshold),
            'precision': float(best_precision),
            'recall': float(best_recall),
            'f1': float(best_f1),
            'support': int(y_true_binary.sum()),
            'is_malignant': i in cfg.MALIGNANT_INDICES
        }
        
        malignant_marker = "⚠️ MALIGNANT" if i in cfg.MALIGNANT_INDICES else ""
        print(f"  {class_name}: threshold={best_threshold:.4f}, P={best_precision:.3f}, R={best_recall:.3f}, F1={best_f1:.3f} {malignant_marker}")
    
    return optimal_thresholds, threshold_metrics


def apply_optimized_thresholds(probs, optimal_thresholds, class_names):
    thresholds_array = np.array([optimal_thresholds[name] for name in class_names])
    normalized_probs = probs / (thresholds_array + 1e-8)
    predictions = np.argmax(normalized_probs, axis=1)
    return predictions


def evaluate_with_optimized_thresholds(oof_df, optimal_thresholds, cfg):
    prob_cols = [f'prob_{i}' for i in range(cfg.N_CLASSES)]
    probs = oof_df[prob_cols].values
    targets = oof_df['target'].values
    
    default_preds = np.argmax(probs, axis=1)
    default_bal_acc = balanced_accuracy_score(targets, default_preds)
    
    optimized_preds = apply_optimized_thresholds(probs, optimal_thresholds, cfg.CLASS_NAMES)
    optimized_bal_acc = balanced_accuracy_score(targets, optimized_preds)
    
    print(f"\n{'='*60}")
    print("COMPARISON: Default vs Optimized Thresholds")
    print(f"{'='*60}")
    print(f"Default Balanced Accuracy: {default_bal_acc:.4f}")
    print(f"Optimized Balanced Accuracy: {optimized_bal_acc:.4f}")
    print(f"Improvement: {(optimized_bal_acc - default_bal_acc)*100:+.2f}%")
    
    return {'default_bal_acc': default_bal_acc, 'optimized_bal_acc': optimized_bal_acc}


# Run threshold optimization
print("\n" + "="*60)
print("STEP: THRESHOLD OPTIMIZATION")
print("="*60)

optimal_thresholds_f1, metrics_f1 = find_optimal_thresholds(oof_df, cfg, strategy='f1')
optimal_thresholds_recall, metrics_recall = find_optimal_thresholds(oof_df, cfg, strategy='recall')

print("\n>>> Evaluating with F1-optimized thresholds:")
eval_results_f1 = evaluate_with_optimized_thresholds(oof_df, optimal_thresholds_f1, cfg)

print("\n>>> Evaluating with Recall-optimized thresholds:")
eval_results_recall = evaluate_with_optimized_thresholds(oof_df, optimal_thresholds_recall, cfg)

# Save thresholds
thresholds_output = {
    'f1_strategy': {'thresholds': optimal_thresholds_f1, 'metrics': metrics_f1, 'balanced_accuracy': eval_results_f1['optimized_bal_acc']},
    'recall_strategy': {'thresholds': optimal_thresholds_recall, 'metrics': metrics_recall, 'balanced_accuracy': eval_results_recall['optimized_bal_acc']}
}

with open(os.path.join(cfg.OUTPUT_DIR, 'optimal_thresholds.json'), 'w') as f:
    json.dump(thresholds_output, f, indent=4)
print(f"\n✓ Saved optimal thresholds")

In [ ]:
def tta_predict(model, image, val_transforms, device, num_tta=4):
    model.eval()
    preds = []
    
    with torch.no_grad():
        img = val_transforms(image=image)["image"].unsqueeze(0).to(device)
        with autocast():
            preds.append(torch.softmax(model(img), dim=1))
        
        if num_tta >= 2:
            img_hflip = val_transforms(image=cv2.flip(image, 1))["image"].unsqueeze(0).to(device)
            with autocast():
                preds.append(torch.softmax(model(img_hflip), dim=1))
        
        if num_tta >= 3:
            img_vflip = val_transforms(image=cv2.flip(image, 0))["image"].unsqueeze(0).to(device)
            with autocast():
                preds.append(torch.softmax(model(img_vflip), dim=1))
        
        if num_tta >= 4:
            img_hvflip = val_transforms(image=cv2.flip(cv2.flip(image, 0), 1))["image"].unsqueeze(0).to(device)
            with autocast():
                preds.append(torch.softmax(model(img_hvflip), dim=1))
    
    return torch.stack(preds).mean(dim=0)


def inference_test(test_csv_path, test_img_root, cfg, use_tta=True, num_tta=4, optimal_thresholds=None):
    test_df = pd.read_csv(test_csv_path)
    val_transforms = get_val_transforms(cfg.IMG_SIZE)
    
    print(f"\n{'='*60}")
    print(f"TEST SET INFERENCE - DenseNet121-CBAM")
    print(f"{'='*60}")
    print(f"Test samples: {len(test_df)}")
    print(f"Using TTA: {use_tta} (num_tta={num_tta})")
    
    # Load models
    models = []
    for fold in range(cfg.N_FOLDS):
        model = create_model(cfg, use_dp=False)
        checkpoint_path = os.path.join(cfg.OUTPUT_DIR, f"densenet121_cbam_fold{fold}_best.pth")
        checkpoint = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        models.append(model)
        print(f"  ✓ Loaded fold {fold} (bal_acc: {checkpoint['best_bal_acc']:.4f})")
    
    all_preds = []
    all_images = []
    
    for idx in tqdm(range(len(test_df)), desc="Inference"):
        row = test_df.iloc[idx]
        img_path = os.path.join(test_img_root, os.path.basename(row["path"]))
        image = cv2.imread(img_path)
        
        if image is None:
            continue
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        all_images.append(row['image'])
        
        fold_preds = []
        for model in models:
            if use_tta:
                pred = tta_predict(model, image, val_transforms, cfg.DEVICE, num_tta=num_tta)
            else:
                img_tensor = val_transforms(image=image)["image"].unsqueeze(0).to(cfg.DEVICE)
                with torch.no_grad(), autocast():
                    pred = torch.softmax(model(img_tensor), dim=1)
            fold_preds.append(pred)
        
        avg_pred = torch.stack(fold_preds).mean(dim=0).cpu().numpy()[0]
        all_preds.append(avg_pred)
    
    all_preds = np.array(all_preds)
    
    if optimal_thresholds is not None:
        pred_classes = apply_optimized_thresholds(all_preds, optimal_thresholds, cfg.CLASS_NAMES)
    else:
        pred_classes = all_preds.argmax(axis=1)
    
    submission = pd.DataFrame({
        'image': all_images,
        **{f'prob_{i}': all_preds[:, i] for i in range(cfg.N_CLASSES)},
        'pred': pred_classes,
        'pred_label': [cfg.CLASS_NAMES[p] for p in pred_classes]
    })
    
    output_path = os.path.join(cfg.OUTPUT_DIR, 'densenet121_cbam_test_predictions.csv')
    submission.to_csv(output_path, index=False)
    print(f"\n✓ Saved predictions to {output_path}")
    
    print(f"\nPrediction Distribution:")
    for i, class_name in enumerate(cfg.CLASS_NAMES):
        count = (pred_classes == i).sum()
        pct = count / len(pred_classes) * 100
        malignant = "⚠️" if i in cfg.MALIGNANT_INDICES else "  "
        print(f"  {malignant} {class_name}: {count} ({pct:.1f}%)")
    
    for model in models:
        del model
    torch.cuda.empty_cache()
    
    return submission


# Run inference
test_submission = inference_test(cfg.TEST_CSV_PATH, cfg.TEST_IMG_ROOT, cfg, use_tta=True, num_tta=4, optimal_thresholds=None)

test_submission_optimized = inference_test(cfg.TEST_CSV_PATH, cfg.TEST_IMG_ROOT, cfg, use_tta=True, num_tta=4, optimal_thresholds=optimal_thresholds_f1)
test_submission_optimized.to_csv(os.path.join(cfg.OUTPUT_DIR, 'densenet121_cbam_test_predictions_optimized.csv'), index=False)

In [ ]:
def plot_confusion_matrix(cm, class_names, save_path, title="Confusion Matrix"):
    plt.figure(figsize=(12, 10))
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f'{cm[i, j]}\n({cm_normalized[i, j]*100:.1f}%)'
    sns.heatmap(cm_normalized, annot=annot, fmt='', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label'); plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✓ Saved {save_path}")


def plot_roc_curves(oof_df, cfg, save_path):
    plt.figure(figsize=(12, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, cfg.N_CLASSES))
    for i, class_name in enumerate(cfg.CLASS_NAMES):
        y_true = (oof_df['target'] == i).astype(int)
        y_prob = oof_df[f'prob_{i}'].values
        try:
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            roc_auc = auc(fpr, tpr)
            linestyle = '--' if i in cfg.MALIGNANT_INDICES else '-'
            label = f'{class_name} (AUC={roc_auc:.3f})' + (' ⚠️' if i in cfg.MALIGNANT_INDICES else '')
            plt.plot(fpr, tpr, color=colors[i], linestyle=linestyle, linewidth=2, label=label)
        except:
            pass
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('ROC Curves - DenseNet121-CBAM', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✓ Saved {save_path}")


def plot_training_history(all_histories, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = plt.cm.Set2(np.linspace(0, 1, len(all_histories)))
    for fold_idx, history in enumerate(all_histories):
        c = colors[fold_idx]
        axes[0, 0].plot(history['epoch'], history['train_loss'], label=f'Fold {fold_idx}', color=c)
        axes[0, 1].plot(history['epoch'], history['val_loss'], label=f'Fold {fold_idx}', color=c)
        axes[1, 0].plot(history['epoch'], history['bal_acc'], label=f'Fold {fold_idx}', color=c)
        axes[1, 1].plot(history['epoch'], history['learning_rate'], label=f'Fold {fold_idx}', color=c)
    for ax, title in zip(axes.flatten(), ['Train Loss', 'Val Loss', 'Balanced Accuracy', 'Learning Rate']):
        ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')
    plt.suptitle('Training History - DenseNet121-CBAM', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"✓ Saved {save_path}")


# Save results
config_dict = {
    'model_name': 'densenet121-cbam',
    'cbam_reduction': cfg.CBAM_REDUCTION,
    'cbam_kernel_size': cfg.CBAM_KERNEL_SIZE,
    'img_size': cfg.IMG_SIZE, 'batch_size': cfg.BATCH_SIZE,
    'base_lr': cfg.BASE_LR, 'drop_rate': cfg.DROP_RATE,
    'use_mixup': cfg.USE_MIXUP, 'mixup_prob': cfg.MIXUP_PROB,
    'malignant_boost': cfg.MALIGNANT_BOOST, 'scc_extra_boost': cfg.SCC_EXTRA_BOOST,
}
with open(os.path.join(cfg.OUTPUT_DIR, 'experiment_config.json'), 'w') as f:
    json.dump(config_dict, f, indent=4)

training_results = {
    'mean_bal_acc': float(np.mean([r['best_bal_acc'] for r in results])),
    'std_bal_acc': float(np.std([r['best_bal_acc'] for r in results])),
    'folds': [{'fold': r['fold'], 'best_bal_acc': float(r['best_bal_acc'])} for r in results]
}
with open(os.path.join(cfg.OUTPUT_DIR, 'training_results.json'), 'w') as f:
    json.dump(training_results, f, indent=4)

# Visualizations
cm = confusion_matrix(oof_df['target'], oof_preds_class)
plot_confusion_matrix(cm, cfg.CLASS_NAMES, os.path.join(cfg.OUTPUT_DIR, 'confusion_matrix.png'), 'Confusion Matrix - DenseNet121-CBAM')
plot_roc_curves(oof_df, cfg, os.path.join(cfg.OUTPUT_DIR, 'roc_curves.png'))
plot_training_history(all_histories, os.path.join(cfg.OUTPUT_DIR, 'training_history.png'))

print(f"\n{'='*60}")
print("ALL RESULTS SAVED SUCCESSFULLY!")
print(f"{'='*60}")